# Alternative Tracing Methods

![AWTT](../../images/alternative_ways_to_trace_0.png)

So far in this module, we've taken a look at the traceable decorator, and how we can use it to set up tracing.

In this lesson, we're going to look at alternative ways in which we can set up tracing, and when you should think about using these different approaches.

## LangChain and LangGraph

If we are using LangChain or LangGraph, all we need to do to set up tracing is to set a few environment variables

![AWTT](../../images/alternative_ways_to_trace_1.png)

In [1]:
# Or you can use a .env file
from dotenv import load_dotenv
load_dotenv()

True

Don't worry too much about our graph implementation here, you can learn more about LangGraph through our LangGraph Academy course!

In [2]:
import os
import nest_asyncio
import operator
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, AnyMessage, get_buffer_string
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display
from typing import List
from typing_extensions import TypedDict, Annotated
from utils import get_vector_db_retriever, RAG_PROMPT

nest_asyncio.apply()

retriever = get_vector_db_retriever()

# --- Crusty Nails Adaptation: Point ChatOpenAI to Groq ($0) ---
llm = ChatOpenAI(
    model="openai/gpt-oss-120b",
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ.get("GROQ_API_KEY"),
    temperature=0
)

# Define Graph state
class GraphState(TypedDict):
    question: str
    messages: Annotated[List[AnyMessage], operator.add]
    documents: List[Document]

# Define Nodes
def retrieve_documents(state: GraphState):
    messages = state.get("messages", [])
    question = state["question"]
    documents = retriever.invoke(f"{get_buffer_string(messages)} {question}")
    return {"documents": documents}

def generate_response(state: GraphState):
    question = state["question"]
    messages = state["messages"]
    documents = state["documents"]
    
    # Rule #5: Token Diet [:600]
    formatted_docs = "\n\n".join(doc.page_content[:600] for doc in documents)
    
    rag_prompt_formatted = RAG_PROMPT.format(context=formatted_docs, conversation=messages, question=question)
    generation = llm.invoke([HumanMessage(content=rag_prompt_formatted)])
    return {"documents": documents, "messages": [HumanMessage(question), generation]}

# Define Graph
graph_builder = StateGraph(GraphState)
graph_builder.add_node("retrieve_documents", retrieve_documents)
graph_builder.add_node("generate_response", generate_response)
graph_builder.add_edge(START, "retrieve_documents")
graph_builder.add_edge("retrieve_documents", "generate_response")
graph_builder.add_edge("generate_response", END)

simple_rag_graph = graph_builder.compile()

# Invoke
question = "How do I set up tracing if I'm using LangChain?"
response = simple_rag_graph.invoke({"question": question}, config={"metadata": {"foo": "bar"}})
print(response["messages"][-1].content)

[1/3] 🔍 Locating union.parquet...
[2/3] 📖 Found 'resources\union.parquet'. Reading parquet into DataFrame...
      Loaded 2187 document chunks from parquet.
      Using text column: 'texts', metadata column: 'metadatas'
[3/3] ⚡ Building local TF-IDF index across 2187 documents...
✅ Retriever ready in 2.80 seconds! Zero API calls, zero web scraping.
1. Install the LangSmith SDK (e.g., `pip install langsmith`) and set your LangChain API key and project name via environment variables (`LANGCHAIN_API_KEY`, `LANGCHAIN_PROJECT`) or `langsmith.init()`.  
2. Add the LangSmith tracer as a callback handler to your LangChain chain or agent (e.g., `from langsmith import tracing; tracing.tracer = tracing.Tracer(); chain = MyChain(callbacks=[tracing.tracer])`).  
3. Run your application; the tracer will automatically log LLM calls, retriever traces, and metadata to your LangSmith workspace for observability.


We're setting up a simple graph in LangGraph. If you want to learn more about LangGraph, I would highly recommend taking a look at our LangGraph Academy course.

You can also pass in metadata or other fields through an optional config

In [3]:
question = "How do I set up tracing if I'm using LangChain?"
simple_rag_graph.invoke({"question": question}, config={"metadata": {"foo": "bar"}})

{'question': "How do I set up tracing if I'm using LangChain?",
 'messages': [HumanMessage(content="How do I set up tracing if I'm using LangChain?", additional_kwargs={}, response_metadata={}),
  AIMessage(content='To enable tracing with LangChain, install the LangSmith SDK (e.g.,\u202f`pip install langsmith`) and set your API key and tracing flag in the environment (`LANGCHAIN_API_KEY=…` and `LANGCHAIN_TRACING_V2=true`). Then add a LangSmith tracer callback to your chain or agent (e.g., `from langchain.callbacks.tracers import LangChainTracer; tracer = LangChainTracer(); chain.run(..., callbacks=[tracer])`). Finally, run your application; the traces will be sent to your LangSmith project for observability.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 261, 'prompt_tokens': 609, 'total_tokens': 870, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 135, 'rejected_prediction_token

##### Let's take a look in LangSmith!

## Tracing Context Manager

In Python, you can use the trace context manager to log traces to LangSmith. This is useful in situations where:

You want to log traces for a specific block of code.
You want control over the inputs, outputs, and other attributes of the trace.
It is not feasible to use a decorator or wrapper.
Any or all of the above.
The context manager integrates seamlessly with the traceable decorator and wrap_openai wrapper, so you can use them together in the same application.

You still need to set your `LANGSMITH_API_KEY` and `LANGSMITH_TRACING`

![AWTT](../../images/alternative_ways_to_trace_2.png)

In [4]:
from langsmith import traceable, trace
from openai import OpenAI
from typing import List
import nest_asyncio
import os
from utils import get_vector_db_retriever

MODEL_PROVIDER = "groq"
MODEL_NAME = "openai/gpt-oss-120b"
APP_VERSION = 1.0
RAG_SYSTEM_PROMPT = """You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the latest question in the conversation. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.
"""

openai_client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ.get("GROQ_API_KEY")
)
nest_asyncio.apply()
retriever = get_vector_db_retriever()


@traceable(run_type="retriever")
def retrieve_documents(question: str):
    return retriever.invoke(question)


# Completed TODO: No @traceable decorator; scoped using context manager instead
def generate_response(question: str, documents):
    formatted_docs = "\n\n".join(doc.page_content[:600] for doc in documents)

    with trace(
        name="Generate Response",
        run_type="chain", 
        inputs={"question": question, "formatted_docs": formatted_docs},
        metadata={"foo": "bar"},
    ) as ls_trace:
        messages = [
            {"role": "system", "content": RAG_SYSTEM_PROMPT},
            {"role": "user", "content": f"Context: {formatted_docs} \n\n Question: {question}"}
        ]
        response = call_openai(messages)
        # Completed TODO: End trace and record output
        ls_trace.end(outputs={"output": response.choices[0].message.content})
        return response


@traceable(run_type="llm")
def call_openai(
    messages: List[dict], model: str = MODEL_NAME, temperature: float = 0.0
):
    return openai_client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    )


@traceable(name="langsmith_rag", run_type="chain")
def langsmith_rag(question: str):
    documents = retrieve_documents(question)
    response = generate_response(question, documents)
    return response.choices[0].message.content


question = "How do I trace with tracing context?"
ai_answer = langsmith_rag(question)
print(ai_answer)

[1/3] 🔍 Locating union.parquet...
[2/3] 📖 Found 'resources\union.parquet'. Reading parquet into DataFrame...
      Loaded 2187 document chunks from parquet.
      Using text column: 'texts', metadata column: 'metadatas'
[3/3] ⚡ Building local TF-IDF index across 2187 documents...
✅ Retriever ready in 0.50 seconds! Zero API calls, zero web scraping.
Pass a `tracing_context` dictionary (usually containing a `run_id` and optionally a `project_name`) to the LangChain call you want to trace—e.g., `agent.run(input, tracing_context={"run_id": my_run.id})`. You can obtain the `run_id` by creating a run with the LangSmith client (`client.create_run(...)`) or by using the context manager `with tracing_context(project="my‑project"):` which sets it automatically. All downstream calls that receive this `tracing_context` will be linked under the same trace in LangSmith.


In [5]:
question = "How do I trace with tracing context?"
ai_answer = langsmith_rag(question)
print(ai_answer)

Create a tracing context (e.g., with `langsmith.tracing_context(project_name="my‑project")` or by calling `langsmith.get_current_run()` to get a `run_id`) and pass that object as the `tracing_context` argument to your LangChain run/agent executor. The context ties all subsequent calls to the same run, so the traces appear under that project/run in LangSmith. No environment variables are required when you supply the context explicitly.


## wrap_openai

The wrap_openai/wrapOpenAI methods in Python/TypeScript allow you to wrap your OpenAI client in order to automatically log traces -- no decorator or function wrapping required! The wrapper works seamlessly with the @traceable decorator or traceable function and you can use both in the same application.

You still need to set your `LANGSMITH_API_KEY` and `LANGSMITH_TRACING`

![AWTT](../../images/alternative_ways_to_trace_3.png)

In [6]:
import os
import openai
from typing import List
import nest_asyncio
from langsmith import traceable
from langsmith.wrappers import wrap_openai
from utils import get_vector_db_retriever

MODEL_PROVIDER = "groq"
MODEL_NAME = "openai/gpt-oss-120b"
APP_VERSION = 1.0
RAG_SYSTEM_PROMPT = """You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the latest question in the conversation. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.
"""

# Completed TODO: Wrap the client
openai_client = wrap_openai(
    openai.Client(
        base_url="https://api.groq.com/openai/v1",
        api_key=os.environ.get("GROQ_API_KEY")
    )
)

nest_asyncio.apply()
retriever = get_vector_db_retriever()


@traceable(run_type="retriever")
def retrieve_documents(question: str):
    return retriever.invoke(question)


@traceable(run_type="chain")
def generate_response(question: str, documents):
    formatted_docs = "\n\n".join(doc.page_content[:600] for doc in documents)
    messages = [
        {"role": "system", "content": RAG_SYSTEM_PROMPT},
        {"role": "user", "content": f"Context: {formatted_docs} \n\n Question: {question}"}
    ]
    return call_openai(messages)


# Completed TODO: No @traceable decorator needed here! wrap_openai handles it.
def call_openai(messages: List[dict]):
    return openai_client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
    )


@traceable(run_type="chain")
def langsmith_rag_with_wrap_openai(question: str):
    documents = retrieve_documents(question)
    response = generate_response(question, documents)
    return response.choices[0].message.content


question = "How do I trace with wrap_openai?"
ai_answer = langsmith_rag_with_wrap_openai(question)
print(ai_answer)

[1/3] 🔍 Locating union.parquet...
[2/3] 📖 Found 'resources\union.parquet'. Reading parquet into DataFrame...
      Loaded 2187 document chunks from parquet.
      Using text column: 'texts', metadata column: 'metadatas'
[3/3] ⚡ Building local TF-IDF index across 2187 documents...
✅ Retriever ready in 0.44 seconds! Zero API calls, zero web scraping.
Import the wrapper, create the client, and wrap it — e.g.:

```python
from openai import OpenAI
from langsmith.wrappers import wrap_openai

client = wrap_openai(OpenAI())   # now all client.chat.completions.create calls are logged
```

After wrapping, use `client.chat.completions.create(...)` as usual; each call is automatically sent to LangSmith as a trace. For additional granularity, decorate your own functions with `@traceable` (or use the `trace` context manager) so their execution appears as nested child runs.


In [7]:
question = "How do I trace with wrap_openai?"
ai_answer = langsmith_rag_with_wrap_openai(question)
print(ai_answer)

To trace OpenAI calls, import the wrapper and wrap the client:

```python
from openai import OpenAI
from langsmith.wrappers import wrap_openai

client = wrap_openai(OpenAI())
```

Now any `client.chat.completions.create(...)` (or other OpenAI SDK calls) will automatically be logged to LangSmith. You can combine this with `@traceable`‑decorated functions or the `trace` context manager for full pipeline tracing.


The wrapped OpenAI client accepts all the same langsmith_extra parameters as @traceable decorated functions

In [8]:
messages = [{"role": "user", "content": "What color is the sky?"}]

openai_client.chat.completions.create(
    model=MODEL_NAME,
    messages=messages,
    langsmith_extra={"metadata": {"foo": "bar"}},
)

ChatCompletion(id='chatcmpl-2b4da78b-736b-41ee-93ec-6abfece94b05', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='The sky is most commonly described as **blue** during a clear daytime. Its color can change dramatically at different times and under different conditions:\n\n- **Sunrise/Sunset:** shades of red, orange, pink, and purple.\n- **Night:** dark black or deep navy, often dotted with stars.\n- **Cloudy or overcast:** gray or white.\n- **Stormy weather:** dark gray to almost black, sometimes with a greenish tint.\n\nSo, while “blue” is the typical answer, the sky’s color varies with the time of day, weather, and atmospheric conditions.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning='The user asks a simple question: "What color is the sky?" Answer straightforwardly: generally blue during day, can be other colors at sunrise/sunset, etc. Provide concise answer.'))], c

## [Advanced] RunTree

Another, more explicit way to log traces to LangSmith is via the RunTree API. This API allows you more control over your tracing - you can manually create runs and children runs to assemble your trace. You still need to set your `LANGSMITH_API_KEY`, but `LANGSMITH_TRACING` is not necessary for this method.

![AWTT](../../images/alternative_ways_to_trace_4.png)

In [9]:
from dotenv import load_dotenv
# I have my env variables defined in a .env file
load_dotenv(override=True)

True

Let's go ahead and set `LANGSMITH_TRACING` to false, as we are using RunTree to manually create runs in this case.

In [10]:
import os
os.environ["LANGSMITH_TRACING"] = "false"

from langsmith import utils
utils.tracing_is_enabled() # This should return false

True

We have rewritten our RAG application, except this time we pass a RunTree argument through our function calls, and create child runs at each layer. This gives our RunTree the same hierarchy that we were automatically able to establish with @traceable

In [11]:
from langsmith import RunTree, utils
print("Tracing enabled?", utils.tracing_is_enabled())  # Should be False

from openai import OpenAI
from typing import List
import nest_asyncio
from utils import get_vector_db_retriever

openai_client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ.get("GROQ_API_KEY")
)
nest_asyncio.apply()
retriever = get_vector_db_retriever()


def retrieve_documents(parent_run: RunTree, question: str):
    child_run = parent_run.create_child(
        name="Retrieve Documents",
        run_type="retriever",
        inputs={"question": question},
    )
    documents = retriever.invoke(question)
    child_run.end(outputs={"documents": [d.page_content[:600] for d in documents]})
    child_run.post()
    return documents


def generate_response(parent_run: RunTree, question: str, documents):
    formatted_docs = "\n\n".join(doc.page_content[:600] for doc in documents)
    rag_system_prompt = """You are an assistant for question-answering tasks. 
    Use the following pieces of retrieved context to answer the latest question in the conversation. 
    If you don't know the answer, just say that you don't know. 
    Use three sentences maximum and keep the answer concise.
    """
    child_run = parent_run.create_child(
        name="Generate Response",
        run_type="chain",
        inputs={"question": question, "formatted_docs": formatted_docs},
    )
    messages = [
        {"role": "system", "content": rag_system_prompt},
        {"role": "user", "content": f"Context: {formatted_docs} \n\n Question: {question}"}
    ]
    openai_response = call_openai(child_run, messages)
    child_run.end(outputs={"output": openai_response.choices[0].message.content})
    child_run.post()
    return openai_response


def call_openai(
    parent_run: RunTree, messages: List[dict], model: str = "openai/gpt-oss-120b", temperature: float = 0.0
):
    child_run = parent_run.create_child(
        name="Groq Call",
        run_type="llm",
        inputs={"messages": messages},
    )
    openai_response = openai_client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    )
    child_run.end(outputs={"response": openai_response.choices[0].message.content})
    child_run.post()
    return openai_response


def langsmith_rag(question: str):
    root_run_tree = RunTree(
        name="Chat Pipeline",
        run_type="chain",
        inputs={"question": question}
    )

    documents = retrieve_documents(root_run_tree, question)
    response = generate_response(root_run_tree, question, documents)
    output = response.choices[0].message.content

    root_run_tree.end(outputs={"generation": output})
    root_run_tree.post()
    return output


question = "How can I trace with RunTree?"
ai_answer = langsmith_rag(question)
print("\nFinal RunTree Answer:\n", ai_answer)

Tracing enabled? True
[1/3] 🔍 Locating union.parquet...
[2/3] 📖 Found 'resources\union.parquet'. Reading parquet into DataFrame...
      Loaded 2187 document chunks from parquet.
      Using text column: 'texts', metadata column: 'metadatas'
[3/3] ⚡ Building local TF-IDF index across 2187 documents...
✅ Retriever ready in 0.45 seconds! Zero API calls, zero web scraping.

Final RunTree Answer:
 You can trace by creating a `RunTree` instance (optionally setting `project_name`, `run_type`, etc.) and then propagating it with the `withRunTree` helper when calling a `traceable` function. For OpenAI calls you can also wrap the client with `langsmith.wrappers.wrap_openai` and pass the same `RunTree` (or just `langsmith_extra` with `project_name`) so the run is automatically recorded in LangSmith. All child runs added inside the `withRunTree` block will be attached to that RunTree and appear in the trace.


In [12]:
question = "How can I trace with RunTree?"
ai_answer = langsmith_rag(question)
print(ai_answer)

You can trace by creating a `RunTree` instance (optionally giving it a `project_name`) and then passing it into traceable calls with the `withRunTree` helper. For example:

```ts
import { RunTree, withRunTree } from "langsmith/traceable";

const rt = new RunTree({ run_type: "chain", project_name: "My Project" });
await withRunTree(rt, () => myTraceableFunction(args));
```

When using OpenAI, wrap the client (`wrappers.wrap_openai`) and include `langsmith_extra: {project_name: "My Project"}` in the request; the wrapper will automatically attach the current `RunTree` and log the trace to LangSmith.
